<a href="https://colab.research.google.com/github/jeosol/llm-post-training/blob/main/rlhf_from_scratch/2_Reward_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reward Model Training

In [ ]:
# Set up the base moel
from transformers import AutoTokenizer
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
%pip install datasets==3.5.0



## Load dataset

In [ ]:
from datasets import load_dataset
dataset_name = 'sst2'
dataset = load_dataset(dataset_name)
dataset

In [ ]:
# View the training and validation dataset
ds_train, ds_val = dataset['train'], dataset['validation']

In [ ]:
ds_train[4]

## Tokenize the dataset

In [ ]:
REWARD_TOKEN_ID = tokenizer.eos_token_id

In [ ]:
REWARD_TOKEN_ID

In [ ]:
text = ds_train[4]['sentence'] #"Hello, this is the first step of RLHF training."
print(text)
tokens = tokenizer(text)
print(tokens)
print(len(tokens['input_ids']))
print(tokenizer.decode(tokens['input_ids']))

In [ ]:
# Function to tokenize a batch
def tokenize(batch):
    outputs = tokenizer(batch['sentence']) # tokenizer returns a dict with keys 'input_ids' and 'attention_mask'
    # create new keys 'score' and 'score_index' for the length of the current batch
    outputs['score'] = [0] * len(outputs['input_ids'])
    outputs['score_index'] = [0] * len(outputs['input_ids'])

    # Loop over the batch
    for i in range(len(outputs['input_ids'])):
        outputs['input_ids'][i].append(REWARD_TOKEN_ID) # extend token array and add REWARD_TOKEN_ID, the eos_token_id
        outputs['attention_mask'][i].append(1) # set attention mask to 1,
        outputs['score'][i] = float(batch['label'][i]) # add the actual label (sentiment) for the data
        outputs['score_index'][i] = len(outputs['input_ids'][i]) - 1 # set the score index for the score,
    return outputs

map_kwargs = {
    "batched": True,
    "batch_size": 512,
    "remove_columns": ['idx', 'sentence', 'label'] # only keep the 'input_ids' and 'attention_mask' columns
}

# create the tokenized dataset
tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

In [ ]:
# view a sample tokenized dataset
tokenized_dataset_train[4]

In [ ]:
# convert the tokenized dataset to tensors
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [ ]:
tokenized_dataset_train[4]

## Filter out shorter tweets

In [ ]:
# include the extra column that holds the score
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 6)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 6)

In [ ]:
len(tokenized_dataset_train)

## LLM with Reward Head

In [ ]:
import torch
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    # How the weights are initialized
    def _post_init(self):
        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1))) # how the weights are initialized
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_head = RewardHead(self.llm.config)

    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1]
        reward = self.reward_head(last_hidden_state).squeeze(-1) # squeeze(-1) removes the last dimension of an array/tensor but only if that dimension has a size of 1
        return torch.sigmoid(reward)

# squeeze(-1) on (1,3,1) will change the shape to (1, 3)
# squeeze(0) on (1,5,4) will become (5,4), and squeeze(-1) is error

In [ ]:
# Create a reward model
model = GPT2RewardHead(model_name)

### Create Dataset Loader

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer)

dataloader_params = {
    'batch_size': 64,
    'shuffle': True,
    'collate_fn': data_collator
}

# create the dataloaders
train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [ ]:
# View one batch of the data
batch = next(iter(train_dataloader))
print(batch.keys())

In [ ]:
print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])

In [ ]:
print(tokenizer.decode(batch['input_ids'][4]))

In [ ]:
batch['attention_mask'][1].nonzero()[-1] # get nonzero indices and return the last one

In [ ]:
batch['attention_mask'][1].nonzero() # torch.nonzero() identifies and returns the indices of all non-zero elements within an input tensor

In [ ]:
outputs = model(batch['input_ids'], batch['attention_mask'])

In [ ]:
print(outputs.shape)

## Training Config

In [ ]:
import torch
import matplotlib.pyplot as plt

#optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
# train for one complete pass of the entire training dataset through the neural network including the both forward and bcakward passes to update model parameters.
num_epochs = 1

# Initialize the storage
batch_losses = []
val_losses = []

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()
num_epochs = 1 # N+ Implementation Detail paper

In [ ]:
def validate():
    model.eval()
    total_loss = 0
    for i, batch in enumerate(val_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        # inference/evaluation - just prediction from the model
        with torch.no_grad():  # torch.no_grad() is a context-manager (or decorator with @torch.no_grad()) that disables the gradient calculation (benefits: reduced memory usage, faster computation (skipping the overhead of tracking and graph-building), prevents "cuda out of memory")
            scores = model(**model_inputs)
            batch_indices = torch.arange(scores.shape[0])
            score = scores[batch_indices, inputs['score_index']]
            target = inputs['score']
            loss = criterion(score, target)
        total_loss += loss.item()
    print('validation loss:', total_loss / len(val_dataloader))

### Training Loop

In [ ]:
# torch.no_grad() - deactivates the gradient engine globally within it's scope to save memory
# model.eval() - only changes the behavior of specific layers like Dropout (turns it off) and Batch Normalization (uses learned statistics)

In [ ]:
model.to(device)

validate()
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        inputs = batch.to(device)
        model_inputs = {
            'input_ids': inputs['input_ids'],
            'attention_mask': inputs['attention_mask']
        }
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
        loss = criterion(score, target)
        print(f'Batch: {i+1}, Loss: {loss.item()}')
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Record the loss for this batch
        batch_losses.append(loss.item())

        #print(loss.item())
    validate()
#Smoothing: Plotting every single batch can result in noisy graph. For a clearer
# trend, you can plot the moving average or record the average loss every N batches

# Plot the Training Loss vs. Batch
plt.plot(batch_losses)
plt.xlabel('Batch Number')
plt.ylabel('Loss')
plt.title('Loss vs. Batch Iterations')
plt.show()

In [ ]:
torch.save(model.state_dict(), 'reward_model.pt')

In [ ]:
validate()

### Confusion Matrix

In [ ]:
# Get a confusion matrix for the validation dataset
from sklearn.metrics import confusion_matrix
model.eval()

all_predictions = []
all_labels = []

for i, batch in enumerate(val_dataloader):
    inputs = batch.to(device)
    model_inputs = {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }
    with torch.no_grad():
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
    predictions = (score > 0.5).int()

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(target.cpu().numpy())

confusion_matrix(all_labels, all_predictions)
